# 03 Graph Metrics Analysis

## Aim
Compute graph-theoretic summaries for the Pearson climate network and test how linear detrending changes the network.

## Method
The notebook loads the saved adjacency matrix, builds a NetworkX graph with latitude-longitude node attributes, computes global and node-level metrics, and compares Pearson networks with and without linear detrending.

## Outputs
- `data/processed/pearson_global_metrics.json`
- `data/processed/pearson_node_metrics.parquet`
- `data/processed/pearson_detrended_*`
- `data/processed/detrending_comparison.json`
- degree, clustering, and link-length figures in `figures/`

## Brief Interpretation
Degree and clustering identify spatially organised regions with many strong links. The detrending comparison checks whether links are driven mainly by shared long-term trends or by shorter-timescale co-variability.


In [6]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

import json

import networkx as nx
import numpy as np
import pandas as pd

from src.config import EDGE_DENSITY, FIGURES_DIR, INTERIM_DATA_DIR
from src.config import PROCESSED_DATA_DIR
from src.dependence import compute_pearson_matrix
from src.graph_metrics import compute_basic_metrics, compute_node_metrics
from src.graph_metrics import save_node_metrics
from src.network_comparison import compare_networks
from src.network_construction import adjacency_from_fixed_density
from src.network_construction import build_graph_from_adjacency
from src.preprocessing import optionally_detrend_timeseries
from src.visualisation import plot_degree_histogram, plot_degree_map
from src.visualisation import plot_link_length_distribution
from src.visualisation import plot_link_length_distributions, plot_metric_map

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


ImportError: cannot import name 'compute_link_lengths_km' from 'src.visualisation' (/Users/jw_wang/Library/CloudStorage/OneDrive-UniversityCollegeLondon/A-docs/UCL/Year 2/Summer project/climate_dynamics/src/visualisation.py)

## Load the Pearson Graph

The graph is reconstructed from the saved adjacency matrix and node metadata. Keeping these outputs separate makes the analysis reproducible without recomputing correlations every time.


In [ ]:
A = np.load(PROCESSED_DATA_DIR / "pearson_adjacency.npy")
node_metadata = pd.read_parquet(INTERIM_DATA_DIR / "node_metadata.parquet")
G = build_graph_from_adjacency(A, node_metadata)
G


## Global Metrics

These metrics describe the graph as a whole: size, density, average degree, clustering, connectedness, and shortest-path length in the largest connected component.


In [ ]:
global_metrics = compute_basic_metrics(G)
(PROCESSED_DATA_DIR / "pearson_global_metrics.json").write_text(
    json.dumps(global_metrics, indent=2, allow_nan=True)
)
global_metrics


## Node Metrics

Node metrics are saved with latitude-longitude attributes, making it possible to map degree, clustering, and centrality back onto the spatial grid.


In [ ]:
node_metrics = compute_node_metrics(G)
save_node_metrics(node_metrics, PROCESSED_DATA_DIR / "pearson_node_metrics.parquet")
node_metrics.head()


## Diagnostic Figures

The first maps show where high-degree or high-clustering nodes occur. The link-length histogram summarises whether retained links are mostly local, long-range, or mixed.


In [ ]:
plot_degree_map(node_metrics, save_path=FIGURES_DIR / "pearson_degree_map.png");
plot_degree_histogram(
    node_metrics,
    save_path=FIGURES_DIR / "pearson_degree_histogram.png",
);
plot_metric_map(
    node_metrics,
    "clustering",
    save_path=FIGURES_DIR / "pearson_clustering_map.png",
);
plot_link_length_distribution(
    G,
    save_path=FIGURES_DIR / "pearson_link_length_distribution.png",
);


## Detrending Comparison

This experiment compares a Pearson network built from the filled anomaly series with one built after removing a linear trend from each node. The comparison reports edge overlap, Jaccard similarity, degree correlation, global clustering, and link-length summaries.


In [ ]:
X = np.load(INTERIM_DATA_DIR / "X_preprocessed.npy")
X_detrended = optionally_detrend_timeseries(X, detrend=True)

corr_detrended = compute_pearson_matrix(X_detrended)
A_detrended = adjacency_from_fixed_density(
    corr_detrended,
    density=EDGE_DENSITY,
    use_absolute=True,
)
G_detrended = build_graph_from_adjacency(A_detrended, node_metadata)

np.save(PROCESSED_DATA_DIR / "pearson_detrended_correlation.npy", corr_detrended)
np.save(PROCESSED_DATA_DIR / "pearson_detrended_adjacency.npy", A_detrended)
nx.write_graphml(
    G_detrended,
    PROCESSED_DATA_DIR / "pearson_detrended_network.graphml",
)

detrended_global_metrics = compute_basic_metrics(G_detrended)
detrended_node_metrics = compute_node_metrics(G_detrended)
(PROCESSED_DATA_DIR / "pearson_detrended_global_metrics.json").write_text(
    json.dumps(detrended_global_metrics, indent=2, allow_nan=True)
)
save_node_metrics(
    detrended_node_metrics,
    PROCESSED_DATA_DIR / "pearson_detrended_node_metrics.parquet",
)

comparison = compare_networks(
    G,
    G_detrended,
    label_a="without_detrending",
    label_b="with_linear_detrending",
)
comparison["global_metrics"] = {
    "without_detrending": global_metrics,
    "with_linear_detrending": detrended_global_metrics,
}
(PROCESSED_DATA_DIR / "detrending_comparison.json").write_text(
    json.dumps(comparison, indent=2, allow_nan=True)
)

plot_link_length_distributions(
    {
        "without detrending": G,
        "with linear detrending": G_detrended,
    },
    save_path=FIGURES_DIR / "detrending_link_length_comparison.png",
);
comparison


## Interpretation

A low Jaccard similarity means detrending changes which grid-point pairs are selected as strongest links. A high degree correlation means that, even if some individual links change, high-degree regions remain broadly similar. Link-length summaries help identify whether detrending preferentially removes local or long-range connections.
